
# Spatial comparison of coastal-flood mangrove and river-flood forest-restoration avoided EADs

This notebook prepares comparable spatial summaries for the Step 4 effectiveness section of paper 3.

It compares:

- coastal-flood avoided EAD attributed to mangrove patches, summarised by parish and by the same major river catchments used in Haggis et al.; and
- river-flood avoided EAD attributed to forest restoration areas from Haggis et al., summarised by parish and by major river catchment.

The notebook does not modify DPhil paper 2 outputs. It reads the river-flood forest-restoration avoided-EAD rasters and the existing mangrove geographical attribution outputs, then writes paper 3 comparison tables under `results/03_cross_hazard_comparison/spatial_service_provision_comparison/`.


In [ ]:

from pathlib import Path
from collections import OrderedDict

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing dphil_papers")


ROOT = find_project_root()
PAPERS = ROOT / "dphil_papers"
PAPER2 = PAPERS / "dphil_paper_2"
PAPER3 = PAPERS / "dphil_paper_3"
COMMON = PAPERS / "dphil_common_cross_cutting"

OUT_DIR = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CRS_METRIC = "EPSG:3448"
J2USD = 1.0 / 150.0

RIVER_EAD_RASTERS = OrderedDict(
    minimum=PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif",
    maximum=PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif",
)

COASTAL_GEO_DIR = PAPER3 / "results_coastal_scenario_comparison" / "weighted_area_distance_signed" / "geographical_mangrove_attribution"
COASTAL_PARISH_FILES = OrderedDict(
    minimum=COASTAL_GEO_DIR / "parish_summary_minimum.csv",
    maximum=COASTAL_GEO_DIR / "parish_summary_maximum.csv",
)
COASTAL_CATCHMENT_FILES = OrderedDict(
    minimum=COASTAL_GEO_DIR / "catchment_summary_minimum.csv",
    maximum=COASTAL_GEO_DIR / "catchment_summary_maximum.csv",
)

PARISH_PATH = COMMON / "common_incoming_data" / "boundaries" / "jam_adm_shp" / "jam_admbnda_adm1.shp"
CATCHMENT_PATH = PAPER2 / "processed_data" / "major_river_catchments" / "major_basins_plus_coastal_unionized_final.gpkg"
CATCHMENT_PARISH_BREAKDOWN_PATH = PAPER2 / "results" / "catchment_attributes" / "catchment_parish_breakdown_all.csv"

for path in [*RIVER_EAD_RASTERS.values(), *COASTAL_PARISH_FILES.values(), *COASTAL_CATCHMENT_FILES.values(), PARISH_PATH, CATCHMENT_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

OUT_DIR



## Load spatial units


In [ ]:

def display_parish_name(name: str) -> str:
    replacements = {
        "Saint Andrew": "St Andrew",
        "Saint Ann": "St Ann",
        "Saint Catherine": "St Catherine",
        "Saint Elizabeth": "St Elizabeth",
        "Saint James": "St James",
        "Saint Mary": "St Mary",
        "Saint Thomas": "St Thomas",
    }
    return replacements.get(name, name)


parishes = gpd.read_file(PARISH_PATH).to_crs(CRS_METRIC)[["NAME_1", "geometry"]].copy()
parishes["parish"] = parishes["NAME_1"].map(display_parish_name)

catchments = gpd.read_file(CATCHMENT_PATH).to_crs(CRS_METRIC)[["catchment_uid", "area_km2", "area_ha", "geometry"]].copy()
catchments["catchment_uid"] = catchments["catchment_uid"].astype(int)

catchment_parish = pd.read_csv(CATCHMENT_PARISH_BREAKDOWN_PATH)
catchment_parish["catchment_uid"] = catchment_parish["catchment_uid"].astype(int)

parishes[["parish"]].head(), catchments[["catchment_uid", "area_km2"]].head()



## River-flood forest restoration avoided EAD by parish and catchment

The river-flood restoration rasters attribute avoided EAD to forest-restoration pixels. Values are stored in JMD and converted to USD using `J2USD = 1/150`, consistent with Haggis et al.


In [ ]:

def zonal_sum_positive_raster_usd(raster_path: Path, zones: gpd.GeoDataFrame, zone_cols: list[str], scenario: str) -> pd.DataFrame:
    with rasterio.open(raster_path) as src:
        arr = src.read(1).astype("float64") * J2USD
        arr[~np.isfinite(arr) | (arr <= 0)] = 0.0
        transform = src.transform
        crs = src.crs
        shape = (src.height, src.width)
        pixel_area_ha = abs(transform.a * transform.e) / 10_000
        raster_total = float(arr.sum())

    zones_ref = zones.to_crs(crs).copy().reset_index(drop=True)
    zones_ref["_zone_id"] = np.arange(1, len(zones_ref) + 1)
    zone_raster = rasterize(
        (
            (geom, int(zone_id))
            for geom, zone_id in zip(zones_ref.geometry, zones_ref["_zone_id"])
            if geom is not None and not geom.is_empty
        ),
        out_shape=shape,
        transform=transform,
        fill=0,
        dtype="int32",
    )

    rows = []
    for _, zone in zones_ref.iterrows():
        mask = zone_raster == int(zone["_zone_id"])
        vals = arr[mask]
        positive = vals > 0
        rec = {col: zone[col] for col in zone_cols}
        rec.update(
            scenario=scenario,
            positive_area_ha=float(positive.sum() * pixel_area_ha),
            avoided_ead_usd=float(vals[positive].sum()),
            raster_total_avoided_ead_usd=raster_total,
        )
        rows.append(rec)

    df = pd.DataFrame(rows)
    assigned_total = df["avoided_ead_usd"].sum()
    df["share_of_assigned_total_pct"] = np.where(assigned_total > 0, df["avoided_ead_usd"] / assigned_total * 100, np.nan)
    df["share_of_raster_total_pct"] = np.where(raster_total > 0, df["avoided_ead_usd"] / raster_total * 100, np.nan)
    df["assigned_total_avoided_ead_usd"] = assigned_total
    df["pct_raster_total_assigned"] = np.where(raster_total > 0, assigned_total / raster_total * 100, np.nan)
    return df.sort_values("avoided_ead_usd", ascending=False).reset_index(drop=True)


river_parish = pd.concat(
    [zonal_sum_positive_raster_usd(path, parishes, ["NAME_1", "parish"], scenario) for scenario, path in RIVER_EAD_RASTERS.items()],
    ignore_index=True,
)
river_catchment = pd.concat(
    [zonal_sum_positive_raster_usd(path, catchments, ["catchment_uid", "area_km2", "area_ha"], scenario) for scenario, path in RIVER_EAD_RASTERS.items()],
    ignore_index=True,
)
river_catchment = river_catchment.merge(
    catchment_parish[["catchment_uid", "parish_breakdown", "majority_parish", "majority_share_pct"]],
    on="catchment_uid",
    how="left",
)

river_parish.head(14), river_catchment.head(12)



## Coastal-flood mangrove avoided EAD by parish and catchment

The coastal summaries are read from the existing geographical mangrove attribution notebook. Net avoided EAD is used here because it matches the coastal-flood effectiveness totals reported in Step 4. Positive and negative attribution components are retained in the output tables.


In [ ]:

def load_coastal_parish(path: Path, scenario: str) -> pd.DataFrame:
    df = pd.read_csv(path).copy()
    df["scenario"] = scenario
    df["parish"] = df["ParishName"].map(display_parish_name)
    df = df.rename(
        columns={
            "ParishName": "parish_raw",
            "MangroveArea_ha": "mangrove_area_ha",
            "Avoided_EAD_USD": "avoided_ead_usd",
            "Positive_Avoided_EAD_USD": "positive_avoided_ead_usd",
            "Negative_Avoided_EAD_USD": "negative_avoided_ead_usd",
            "PctOfTotalMangroveArea": "pct_total_mangrove_area",
            "MangroveCount": "mangrove_patch_count",
        }
    )
    total = df["avoided_ead_usd"].sum()
    df["assigned_total_avoided_ead_usd"] = total
    df["share_of_assigned_total_pct"] = np.where(total != 0, df["avoided_ead_usd"] / total * 100, np.nan)
    return df.sort_values("avoided_ead_usd", ascending=False).reset_index(drop=True)


def load_coastal_catchment(path: Path, scenario: str) -> pd.DataFrame:
    df = pd.read_csv(path).copy()
    df["scenario"] = scenario
    df = df.rename(
        columns={
            "MangroveArea_ha": "mangrove_area_ha",
            "Avoided_EAD_USD": "avoided_ead_usd",
            "Positive_Avoided_EAD_USD": "positive_avoided_ead_usd",
            "Negative_Avoided_EAD_USD": "negative_avoided_ead_usd",
            "PctOfScenarioTotalMangroveArea": "pct_total_mangrove_area_assigned_to_catchments",
            "MangroveCount_Intersecting": "mangrove_patch_count_intersecting",
        }
    )
    total = df["avoided_ead_usd"].sum()
    df["assigned_total_avoided_ead_usd"] = total
    df["share_of_assigned_total_pct"] = np.where(total != 0, df["avoided_ead_usd"] / total * 100, np.nan)
    df = df.merge(
        catchment_parish[["catchment_uid", "parish_breakdown", "majority_parish", "majority_share_pct"]],
        on="catchment_uid",
        how="left",
    )
    return df.sort_values("avoided_ead_usd", ascending=False).reset_index(drop=True)


coastal_parish = pd.concat(
    [load_coastal_parish(path, scenario) for scenario, path in COASTAL_PARISH_FILES.items()],
    ignore_index=True,
)
coastal_catchment = pd.concat(
    [load_coastal_catchment(path, scenario) for scenario, path in COASTAL_CATCHMENT_FILES.items()],
    ignore_index=True,
)

coastal_parish.head(14), coastal_catchment.head(12)



## Combine minimum and maximum scenarios for manuscript tables


In [ ]:

def min_max_table(df: pd.DataFrame, key_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    pieces = []
    for scenario in ["minimum", "maximum"]:
        part = df.loc[df["scenario"] == scenario, key_cols + value_cols].copy()
        part = part.rename(columns={col: f"{col}_{scenario}" for col in value_cols})
        pieces.append(part)
    out = pieces[0].merge(pieces[1], on=key_cols, how="outer")
    return out


river_parish_minmax = min_max_table(
    river_parish,
    ["NAME_1", "parish"],
    ["positive_area_ha", "avoided_ead_usd", "share_of_assigned_total_pct"],
).sort_values("avoided_ead_usd_maximum", ascending=False)

river_catchment_minmax = min_max_table(
    river_catchment,
    ["catchment_uid", "area_km2", "area_ha", "parish_breakdown", "majority_parish", "majority_share_pct"],
    ["positive_area_ha", "avoided_ead_usd", "share_of_assigned_total_pct"],
).sort_values("avoided_ead_usd_maximum", ascending=False)

coastal_parish_minmax = min_max_table(
    coastal_parish,
    ["parish_raw", "parish"],
    ["mangrove_area_ha", "avoided_ead_usd", "positive_avoided_ead_usd", "negative_avoided_ead_usd", "share_of_assigned_total_pct", "pct_total_mangrove_area"],
).sort_values("avoided_ead_usd_maximum", ascending=False)

coastal_catchment_minmax = min_max_table(
    coastal_catchment,
    ["catchment_uid", "area_km2", "area_ha", "parish_breakdown", "majority_parish", "majority_share_pct"],
    ["mangrove_area_ha", "avoided_ead_usd", "positive_avoided_ead_usd", "negative_avoided_ead_usd", "share_of_assigned_total_pct", "pct_total_mangrove_area_assigned_to_catchments"],
).sort_values("avoided_ead_usd_maximum", ascending=False)

river_parish_minmax.head(14)


In [ ]:

coastal_parish_minmax.head(14)


In [ ]:

river_catchment_minmax.head(12)


In [ ]:

coastal_catchment_minmax.head(12)



## Concentration metrics


In [ ]:

def concentration_metrics(df: pd.DataFrame, label_col: str, value_col: str, scenario: str, top_ns=(1, 2, 3, 5, 10)) -> pd.DataFrame:
    sub = df.loc[df["scenario"] == scenario].sort_values(value_col, ascending=False).reset_index(drop=True).copy()
    total = sub[value_col].sum()
    rows = []
    for n in top_ns:
        top = sub.head(n)
        rows.append(
            {
                "scenario": scenario,
                "top_n": n,
                "top_units": "; ".join(top[label_col].astype(str).tolist()),
                "avoided_ead_usd": top[value_col].sum(),
                "share_pct": top[value_col].sum() / total * 100 if total else np.nan,
            }
        )
    return pd.DataFrame(rows)


concentration = pd.concat(
    [
        concentration_metrics(coastal_parish, "parish", "avoided_ead_usd", scenario).assign(hazard="coastal_flood", unit="parish")
        for scenario in ["minimum", "maximum"]
    ]
    + [
        concentration_metrics(river_parish, "parish", "avoided_ead_usd", scenario).assign(hazard="river_flood", unit="parish")
        for scenario in ["minimum", "maximum"]
    ]
    + [
        concentration_metrics(coastal_catchment, "catchment_uid", "avoided_ead_usd", scenario).assign(hazard="coastal_flood", unit="catchment")
        for scenario in ["minimum", "maximum"]
    ]
    + [
        concentration_metrics(river_catchment, "catchment_uid", "avoided_ead_usd", scenario).assign(hazard="river_flood", unit="catchment")
        for scenario in ["minimum", "maximum"]
    ],
    ignore_index=True,
)

concentration[["hazard", "unit", "scenario", "top_n", "top_units", "avoided_ead_usd", "share_pct"]]



## Write outputs and manuscript summary


In [ ]:

def write_csv(df: pd.DataFrame, filename: str) -> Path:
    path = OUT_DIR / filename
    df.to_csv(path, index=False)
    return path


written = []
written.append(write_csv(river_parish, "river_flood_restoration_avoided_ead_by_parish_long.csv"))
written.append(write_csv(river_catchment, "river_flood_restoration_avoided_ead_by_catchment_long.csv"))
written.append(write_csv(coastal_parish, "coastal_flood_mangrove_avoided_ead_by_parish_long.csv"))
written.append(write_csv(coastal_catchment, "coastal_flood_mangrove_avoided_ead_by_catchment_long.csv"))
written.append(write_csv(river_parish_minmax, "river_flood_restoration_avoided_ead_by_parish_minmax.csv"))
written.append(write_csv(river_catchment_minmax, "river_flood_restoration_avoided_ead_by_catchment_minmax.csv"))
written.append(write_csv(coastal_parish_minmax, "coastal_flood_mangrove_avoided_ead_by_parish_minmax.csv"))
written.append(write_csv(coastal_catchment_minmax, "coastal_flood_mangrove_avoided_ead_by_catchment_minmax.csv"))
written.append(write_csv(concentration, "coastal_river_spatial_concentration_metrics.csv"))

written


In [ ]:
def fmt_pct(value: float, digits: int = 1) -> str:
    return f"{value:.{digits}f}%"


def scenario_pair(df: pd.DataFrame, unit: str, hazard: str, top_n: int) -> tuple[pd.Series, pd.Series]:
    rows = df[(df["unit"] == unit) & (df["hazard"] == hazard) & (df["top_n"] == top_n)]
    return rows.loc[rows["scenario"] == "minimum"].iloc[0], rows.loc[rows["scenario"] == "maximum"].iloc[0]


coastal_parish_top1_min, coastal_parish_top1_max = scenario_pair(concentration, "parish", "coastal_flood", 1)
coastal_parish_top2_min, coastal_parish_top2_max = scenario_pair(concentration, "parish", "coastal_flood", 2)
coastal_catch_top2_min, coastal_catch_top2_max = scenario_pair(concentration, "catchment", "coastal_flood", 2)
coastal_catch_top3_min, coastal_catch_top3_max = scenario_pair(concentration, "catchment", "coastal_flood", 3)

river_parish_top1_min, river_parish_top1_max = scenario_pair(concentration, "parish", "river_flood", 1)
river_parish_top3_min, river_parish_top3_max = scenario_pair(concentration, "parish", "river_flood", 3)
river_catch_top2_min, river_catch_top2_max = scenario_pair(concentration, "catchment", "river_flood", 2)
river_catch_top3_min, river_catch_top3_max = scenario_pair(concentration, "catchment", "river_flood", 3)

coastal_catch_total_min = coastal_catchment.loc[coastal_catchment["scenario"] == "minimum", "avoided_ead_usd"].sum()
coastal_catch_total_max = coastal_catchment.loc[coastal_catchment["scenario"] == "maximum", "avoided_ead_usd"].sum()
coastal_parish_total_min = coastal_parish.loc[coastal_parish["scenario"] == "minimum", "avoided_ead_usd"].sum()
coastal_parish_total_max = coastal_parish.loc[coastal_parish["scenario"] == "maximum", "avoided_ead_usd"].sum()
coastal_catch_coverage_min = coastal_catch_total_min / coastal_parish_total_min * 100
coastal_catch_coverage_max = coastal_catch_total_max / coastal_parish_total_max * 100

summary_text = f"""# Spatial service-provision comparison

Parish and catchment summaries show that coastal-flood benefits from mangroves are more spatially concentrated than river-flood benefits from forest restoration. At parish scale, St Catherine alone accounted for {fmt_pct(coastal_parish_top1_min['share_pct'])} and {fmt_pct(coastal_parish_top1_max['share_pct'])} of net mangrove-attributed coastal-flood avoided EAD in the minimum and maximum scenarios, respectively, while St Catherine and Clarendon together accounted for {fmt_pct(coastal_parish_top2_min['share_pct'])} and {fmt_pct(coastal_parish_top2_max['share_pct'])}. For river flooding, no single parish accounted for more than {fmt_pct(max(river_parish_top1_min['share_pct'], river_parish_top1_max['share_pct']))} of forest-restoration avoided EAD, and the top three parishes accounted for {fmt_pct(river_parish_top3_min['share_pct'])} and {fmt_pct(river_parish_top3_max['share_pct'])} in the minimum and maximum scenarios, respectively. This indicates that river-flood restoration benefits are distributed across a broader set of parishes, including St Elizabeth, St Catherine and St Ann.

The same contrast is visible when both hazards are summarised by the major river catchments used by Haggis et al. The catchment overlay captured {fmt_pct(coastal_catch_coverage_min)} and {fmt_pct(coastal_catch_coverage_max)} of national net mangrove-attributed coastal-flood avoided EAD in the minimum and maximum scenarios, respectively. Within this assigned catchment total, the two largest catchments for coastal-flood mangrove benefits, catchments {coastal_catch_top2_min['top_units'].replace('; ', ' and ')}, accounted for {fmt_pct(coastal_catch_top2_min['share_pct'])} and {fmt_pct(coastal_catch_top2_max['share_pct'])} in the minimum and maximum scenarios, respectively; the top three accounted for {fmt_pct(coastal_catch_top3_min['share_pct'])} and {fmt_pct(coastal_catch_top3_max['share_pct'])}. By contrast, for river flooding, the two largest catchments, catchments {river_catch_top2_min['top_units'].replace('; ', ' and ')}, accounted for {fmt_pct(river_catch_top2_min['share_pct'])} and {fmt_pct(river_catch_top2_max['share_pct'])}, and the top three for {fmt_pct(river_catch_top3_min['share_pct'])} and {fmt_pct(river_catch_top3_max['share_pct'])}. This suggests that mangrove-based coastal protection is strongly dependent on a small number of coastal systems associated with the Kingston-Portmore-St Catherine corridor, whereas river-flood benefits from forest restoration are spatially more dispersed across catchments.
"""

summary_path = OUT_DIR / "spatial_service_provision_comparison_written_summary.md"
summary_path.write_text(summary_text)
print(summary_text)
print("\nWrote: " + str(summary_path))